# FraudFlow Week 7: Random Forest, OOB & Permutation Importance

This notebook covers the full Week 7 pipeline:
- Data loading and canonical split
- Preprocessing
- Random Forest training (n_estimators=150, balanced, oob_score=True)
- OOB evaluation
- Final test set evaluation
- Confusion matrix, PR curve, ROC curve
- Impurity feature importance
- Permutation importance
- Leakage checks
- Interpretation

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# â”€â”€ 1. Data Loading â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Data Loading ===')
df_raw = pd.read_csv('../data/raw/fraudTest.csv')
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Fraud rate: {df_raw["is_fraud"].mean():.4%}')
df_raw.head(3)

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from sklearn.model_selection import train_test_split
from src.preprocessing.feature_engineering import engineer_features
from src.preprocessing.pipeline import build_preprocessor

# â”€â”€ 2. Feature Engineering & Canonical Split â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Feature Engineering ===')
df_clf = engineer_features(df_raw)
y = df_clf['is_fraud']
X = df_clf.drop(columns=['is_fraud'], errors='ignore')

print('=== Canonical Train/Test Split (stratified, random_state=42) ===')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape[0]:,} samples, {y_train.sum():,} fraud ({y_train.mean():.4%})')
print(f'Test:  {X_test.shape[0]:,} samples, {y_test.sum():,} fraud ({y_test.mean():.4%})')

In [ ]:
# â”€â”€ 3. Preprocessing (fit on train only) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Preprocessing ===')
preprocessor = build_preprocessor(X_train)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)  # transform only â€” no fit on test!
print(f'Processed train shape: {X_train_proc.shape}')
print(f'Processed test shape:  {X_test_proc.shape}')

In [ ]:
import joblib

# â”€â”€ 4. Random Forest Model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Load the pre-trained artifact (trained by scripts/run_week7.py)
rf = joblib.load('../models/random_forest_week7.joblib')
print('=== Random Forest Configuration ===')
print(f'  n_estimators   : {rf.n_estimators}')
print(f'  random_state   : {rf.random_state}')
print(f'  class_weight   : {rf.class_weight}')
print(f'  oob_score      : {rf.oob_score}')
print(f'  max_depth      : {rf.max_depth}')
print(f'  min_samples_leaf: {rf.min_samples_leaf}')
print(f'  n_jobs         : {rf.n_jobs}')

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)

# â”€â”€ 5. OOB Evaluation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== OOB Evaluation ===')
print(f'  OOB Accuracy (rf.oob_score_): {rf.oob_score_:.4f}')

oob_probs = rf.oob_decision_function_[:, 1]
mask = ~np.isnan(oob_probs)
y_train_arr = y_train.values
oob_pr_auc = average_precision_score(y_train_arr[mask], oob_probs[mask])
oob_roc_auc = roc_auc_score(y_train_arr[mask], oob_probs[mask])
print(f'  OOB PR-AUC   : {oob_pr_auc:.4f}')
print(f'  OOB ROC-AUC  : {oob_roc_auc:.4f}')

In [ ]:
# â”€â”€ 6. Final Test Evaluation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Final Test Set Evaluation (threshold = 0.5) ===')
test_preds = rf.predict(X_test_proc)
test_probs = rf.predict_proba(X_test_proc)[:, 1]

test_metrics = {
    'Accuracy' : accuracy_score(y_test, test_preds),
    'Precision': precision_score(y_test, test_preds, zero_division=0),
    'Recall'   : recall_score(y_test, test_preds, zero_division=0),
    'F1'       : f1_score(y_test, test_preds, zero_division=0),
    'PR-AUC'   : average_precision_score(y_test, test_probs),
    'ROC-AUC'  : roc_auc_score(y_test, test_probs),
}
for k, v in test_metrics.items():
    print(f'  {k:10s}: {v:.4f}')

In [ ]:
# â”€â”€ 7. Metrics report â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Full Metrics Report (phase7_metrics.json) ===')
with open('../reports/phase7_metrics.json', 'r') as f:
    phase7_metrics = json.load(f)
print(json.dumps(phase7_metrics, indent=2))

In [ ]:
# â”€â”€ 8. Confusion Matrix â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(7, 5))
img = mpimg.imread('../reports/figures/phase7/confusion_matrix.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ 9. Precision-Recall Curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(mpimg.imread('../reports/figures/phase7/precision_recall_curve.png'))
axes[0].axis('off')
axes[0].set_title('Precision-Recall Curve')
axes[1].imshow(mpimg.imread('../reports/figures/phase7/roc_curve.png'))
axes[1].axis('off')
axes[1].set_title('ROC Curve')
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ 10. OOB vs Test Comparison â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(mpimg.imread('../reports/figures/phase7/oob_vs_test_metrics.png'))
ax.axis('off')
ax.set_title('OOB vs Test Metrics Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ 11. Impurity Feature Importance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Top 10 Impurity Feature Importances ===')
feat_df = pd.read_csv('../reports/phase7_feature_importance.csv')
print(feat_df.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread('../reports/figures/phase7/feature_importance.png'))
ax.axis('off')
ax.set_title('Impurity Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
# â”€â”€ 12. Permutation Importance â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('=== Top 10 Permutation Importances (PR-AUC decrease) ===')
perm_df = pd.read_csv('../reports/phase7_permutation_importance.csv')
print(perm_df.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread('../reports/figures/phase7/permutation_importance.png'))
ax.axis('off')
ax.set_title('Permutation Importance (Top 20)')
plt.tight_layout()
plt.show()

## Interpretation

### Two Importance Methods Compared

| Method | Basis | Data | Bias |
|---|---|---|---|
| **Impurity (Gini)** | Mean Gini decrease across all trees | Training data | Biased toward high-cardinality features |
| **Permutation** | Drop in PR-AUC when feature shuffled | Test data subset | More realistic; directly linked to metric |

### Leakage Checks
- âœ… Raw dataset untouched (`data/raw/fraudTest.csv` â€” 555,719 Ã— 23)
- âœ… Preprocessor **fit on training set only**, transform applied to test set
- âœ… Test set not seen during training or model selection
- âœ… Permutation importance computed on test set post-training
- âœ… Threshold fixed at 0.5 â€” no threshold tuning performed

### Limitations
- Random Forest is memory-intensive (~13 MB for this configuration)
- OOB evaluation uses ~36.8% of training samples not in each tree's bootstrap
- Permutation importance computed on a 1,000-sample subset for feasibility
- Model is a black-box â€” individual predictions cannot be easily explained without further tools